# Comprensión y Análisis Exploratorio de Datos (EDA)

Este notebook realiza la exploración inicial y el análisis exploratorio 
del dataset de créditos, como paso previo al modelado. Se busca entender 
la estructura de los datos, detectar problemas de calidad (nulos, tipos 
incorrectos) y comprender las relaciones entre variables, especialmente 
con la variable objetivo `Pago_atiempo`.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("../../Base_de_datos.csv")
df.head()

Matplotlib is building the font cache; this may take a moment.


,tipo_credito,fecha_prestamo,capital_prestado,plazo_meses,edad_cliente,tipo_laboral,salario_cliente,total_otros_prestamos,cuota_pactada,puntaje,...,saldo_mora,saldo_total,saldo_principal,saldo_mora_codeudor,creditos_sectorFinanciero,creditos_sectorCooperativo,creditos_sectorReal,promedio_ingresos_datacredito,tendencia_ingresos,Pago_atiempo
0,7,2024-12-21 11:31:35,3692160.0,10,42,Independiente,8000000,2500000,341296,88.768094,...,0.0,51258.0,51258.0,0.0,5,0,0,908526.0,Estable,1
1,4,2025-04-22 09:47:35,840000.0,6,60,Empleado,3000000,2000000,124876,95.227787,...,0.0,8673.0,8673.0,0.0,0,0,2,939017.0,Creciente,1
2,9,2026-01-08 12:22:40,5974028.4,10,36,Independiente,4036000,829000,529554,47.613894,...,0.0,18702.0,18702.0,0.0,3,0,0,NaN,NaN,0
3,4,2025-08-04 12:04:10,1671240.0,6,48,Empleado,1524547,498000,252420,95.227787,...,0.0,15782.0,15782.0,0.0,3,0,0,1536193.0,Creciente,1
4,9,2025-04-26 11:24:26,2781636.0,11,44,Empleado,5000000,4000000,217037,95.227787,...,0.0,204804.0,204804.0,0.0,3,0,1,933473.0,Creciente,1


## 1. Exploración inicial de datos

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10763 entries, 0 to 10762
Data columns (total 23 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   tipo_credito                   10763 non-null  int64  
 1   fecha_prestamo                 10763 non-null  str    
 2   capital_prestado               10763 non-null  float64
 3   plazo_meses                    10763 non-null  int64  
 4   edad_cliente                   10763 non-null  int64  
 5   tipo_laboral                   10763 non-null  str    
 6   salario_cliente                10763 non-null  int64  
 7   total_otros_prestamos          10763 non-null  int64  
 8   cuota_pactada                  10763 non-null  int64  
 9   puntaje                        10763 non-null  float64
 10  puntaje_datacredito            10757 non-null  float64
 11  cant_creditosvigentes          10763 non-null  int64  
 12  huella_consulta                10763 non-null  int64  
 1

In [3]:
df.isnull().sum()

tipo_credito                        0
fecha_prestamo                      0
capital_prestado                    0
plazo_meses                         0
edad_cliente                        0
tipo_laboral                        0
salario_cliente                     0
total_otros_prestamos               0
cuota_pactada                       0
puntaje                             0
puntaje_datacredito                 6
cant_creditosvigentes               0
huella_consulta                     0
saldo_mora                        156
saldo_total                       156
saldo_principal                   405
saldo_mora_codeudor               590
creditos_sectorFinanciero           0
creditos_sectorCooperativo          0
creditos_sectorReal                 0
promedio_ingresos_datacredito    2930
tendencia_ingresos               2932
Pago_atiempo                        0
dtype: int64

### Caracterización de las variables

**Variables numéricas continuas:** `capital_prestado`, `puntaje`, `puntaje_datacredito`, 
`saldo_mora`, `saldo_total`, `saldo_principal`, `saldo_mora_codeudor`, `promedio_ingresos_datacredito`

**Variables numéricas discretas:** `plazo_meses`, `edad_cliente`, `salario_cliente`, 
`total_otros_prestamos`, `cuota_pactada`, `cant_creditosvigentes`, `huella_consulta`, 
`creditos_sectorFinanciero`, `creditos_sectorCooperativo`, `creditos_sectorReal`

**Variables categóricas nominales:** `tipo_credito` (código numérico que representa una categoría, 
no una cantidad), `tipo_laboral` (Independiente/Empleado), `tendencia_ingresos` (Estable/Creciente/Decreciente)

**Variable dicotómica (binaria):** `Pago_atiempo` — es nuestra **variable objetivo** (0 = no pagó a 
tiempo, 1 = sí pagó a tiempo)

**Variable de fecha:** `fecha_prestamo` — actualmente está como texto (`str`/`object`), hay que 
convertirla a formato fecha (`datetime`).

### Revisión de nulos

Encontramos valores nulos en varias columnas. Las que más preocupan son:
- `promedio_ingresos_datacredito`: 2930 nulos (~27% del dataset)
- `tendencia_ingresos`: 2932 nulos (~27% del dataset)
- `saldo_mora_codeudor`: 590 nulos (~5.5%)

El resto (`puntaje_datacredito`, `saldo_mora`, `saldo_total`, `saldo_principal`) tiene una 
proporción baja de nulos (menor al 4%). Más adelante vamos a decidir cómo tratarlos 
(imputación, eliminación, o creación de una categoría "Sin dato").

**corregir el tipo de dato de la fecha**

In [4]:
df["fecha_prestamo"] = pd.to_datetime(df["fecha_prestamo"])
df["fecha_prestamo"].dtype

dtype('<M8[us]')

### Investigación de `tendencia_ingresos`

In [5]:
df["tendencia_ingresos"].unique()

<ArrowStringArray>
[    'Estable',   'Creciente',           nan, 'Decreciente',        '8315',
           '0',      '158042',        '3978',        '9147',      '168750',
      '-28589',     '1000000',     '-566272',       '24702',       '31837',
      '122727',      '417087',        '9090',      '173031',      '-70715',
     '-435177',     '-702927',       '-4105',       '54683',       '22832',
      '209090',        '5697',       '10808',        '-288',     '-164315',
     '2029000',       '17181',       '15245',       '82657',       '52862',
     '1817052',       '75761',      '146918',     '1123000',       '15090',
     '4250635',       '22363',     '-101368',       '86286',       '65988',
       '77975',     '-224714']
Length: 47, dtype: str

In [6]:
categorias_validas = ["Estable", "Creciente", "Decreciente"]
valores_invalidos = df[~df["tendencia_ingresos"].isin(categorias_validas) & df["tendencia_ingresos"].notna()]
print(f"Cantidad de valores inválidos: {len(valores_invalidos)}")
valores_invalidos[["tendencia_ingresos", "promedio_ingresos_datacredito"]].head(10)

Cantidad de valores inválidos: 58


,tendencia_ingresos,promedio_ingresos_datacredito
157,8315,939017.0
168,0,1200000.0
440,158042,28878675.0
486,3978,1562215.0
705,9147,1000000.0
1185,8315,977131.0
1188,168750,2550000.0
1898,-28589,5425531.0
2027,8315,916148.0
2038,1000000,917014.0


Se investigó si los 58 valores "inválidos" de `tendencia_ingresos` (números en vez de categorías) 
tenían relación con `promedio_ingresos_datacredito` de la misma fila, pero no se encontró una 
correspondencia clara ni consistente (incluso hay valores negativos, lo cual no tiene sentido para 
un ingreso). Se concluye que es un error de calidad de datos sin forma de recuperar el valor original, 
por lo que estos 58 registros se van a tratar como valores nulos.

In [7]:
df.loc[~df["tendencia_ingresos"].isin(categorias_validas), "tendencia_ingresos"] = np.nan
df["tendencia_ingresos"].unique()

<ArrowStringArray>
['Estable', 'Creciente', nan, 'Decreciente']
Length: 4, dtype: str

### Eliminación de variables irrelevantes

Revisamos la cantidad de valores únicos por columna, para detectar posibles identificadores 
(alta cardinalidad, un valor distinto por fila) o columnas constantes (un solo valor en todo 
el dataset), que no aportarían información útil al modelo.

In [8]:
df.nunique().sort_values()

tipo_laboral                         2
Pago_atiempo                         2
tendencia_ingresos                   3
saldo_mora_codeudor                  4
tipo_credito                         6
creditos_sectorCooperativo          11
plazo_meses                         18
creditos_sectorReal                 24
huella_consulta                     28
creditos_sectorFinanciero           33
cant_creditosvigentes               39
edad_cliente                        54
saldo_mora                          55
puntaje                            248
puntaje_datacredito                315
salario_cliente                   1385
total_otros_prestamos             1538
promedio_ingresos_datacredito     5309
capital_prestado                  7306
saldo_principal                   8647
saldo_total                       8858
cuota_pactada                     9836
fecha_prestamo                   10758
dtype: int64

**Conclusión — variables irrelevantes:** no se encontraron columnas constantes (un solo valor) 
ni identificadores únicos evidentes en el dataset. La columna `fecha_prestamo` tiene una 
cardinalidad muy alta (10758 valores únicos sobre 10763 filas), por lo que en su forma actual 
(fecha con hora exacta) no aporta un patrón generalizable para el modelo. No se elimina en 
esta etapa: se transformará más adelante, derivando atributos como mes o año, en vez de 
usarse tal cual.

**convertir tipo_credito a categoría**

In [9]:
df["tipo_credito"] = df["tipo_credito"].astype("category")
df["tipo_credito"].dtype

CategoricalDtype(categories=[4, 6, 7, 9, 10, 68], ordered=False, categories_dtype=int64)

In [10]:
df["tipo_laboral"] = df["tipo_laboral"].astype("category")
df["tendencia_ingresos"] = df["tendencia_ingresos"].astype("category")

df[["tipo_laboral", "tendencia_ingresos"]].dtypes

tipo_laboral          category
tendencia_ingresos    category
dtype: object